# 🩺 Smart Disease Prediction and Patient Support System

**Programming for AI — Group Project**

This notebook is self-contained and designed to run top-to-bottom in **Google Colab**
(`Runtime ▸ Run all`). It covers the full pipeline for four AI/ML components built on
public Kaggle healthcare datasets:

1. **Classification** — predict a disease from a structured set of symptoms (RandomForest).
2. **Natural Language Processing** — predict a disease from a free-text symptom description
   (TF-IDF + Logistic Regression).
3. **Deep Learning** — recognise a visible skin condition from an uploaded photo
   (transfer-learned ResNet18).
4. **Explainable AI (SHAP)** — explain *why* the classification model made a given prediction.
5. **Sentiment Analysis** — classify patient feedback / drug reviews as Positive, Neutral or
   Negative (TF-IDF + Logistic Regression).

All charts and tables below are produced directly as **cell outputs** — nothing is written
to disk. The final section is a small, self-contained **inference demo** (Colab form fields
+ file upload) that replaces the previous Streamlit web app — no separate application to
deploy or run.

> ⚠️ Educational prototype only — not a medical device. Predictions are for demonstration
> purposes as part of a university assessment.

**Before running:** you need a Kaggle account and API token (`kaggle.json`) — see
[kaggle.com/settings](https://www.kaggle.com/settings) ▸ *API* ▸ *Create New Token*. You'll
be prompted to upload it in Section 2.


## 1. Setup

In [ ]:
# Installed explicitly so the notebook is self-contained and doesn't depend on what
# happens to already be in the Colab image. torch/torchvision are the one exception:
# Colab preinstalls a build matched to its GPU driver, and reinstalling via pip can
# silently break GPU acceleration -- so we leave those two as Colab provides them.
# (If you're running outside Colab, also: pip install torch torchvision)
!pip install -q kaggle shap joblib scikit-learn pandas numpy matplotlib seaborn pillow


In [ ]:
import html
import io
import json
import re
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")


In [ ]:
ROOT = Path("/content")
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

for d in (RAW_DIR, PROCESSED_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Kaggle dataset refs -> local subfolder under data/raw
DATASETS = {
    "classification": "itachi9604/disease-symptom-description-dataset",
    "nlp": "niyarrbarman/symptom2disease",
    "images": "riyaelizashaju/skin-disease-classification-image-dataset",
    "sentiment": "jessicali9530/kuc-hackathon-winter-2018",
}

RANDOM_STATE = 42

CONFIDENCE_THRESHOLDS = {
    "classification": 0.45,
    "nlp": 0.45,
    "sentiment": 0.60,
    "images": 0.60,
}


MIN_IMAGE_STD = 5.0

ACCEPTED_ACCURACY = {
    "classification": 0.85,
    "nlp": 0.80,
    "sentiment": 0.55,
    "images": 0.55,
}


## 2. Download datasets from Kaggle

Fill in your Kaggle credentials below, then run the cell. Get them from
[kaggle.com/settings](https://www.kaggle.com/settings) ▸ *API* ▸ *Create New Token* (this
downloads a `kaggle.json` containing both values). Re-running this section is safe —
datasets already downloaded are skipped.

> ⚠️ **Don't commit real credentials.** This assessment requires a public GitHub repo —
> if you push this notebook with your real key filled in, it becomes public too. Clear
> the two values back to placeholders before committing, or better, use
> [Colab Secrets](https://colab.research.google.com) (key icon in the left sidebar) so
> the values never live in the notebook file at all.


In [ ]:
# Get these from https://www.kaggle.com/settings -> API -> "Create New Token"
KAGGLE_USERNAME = "sunny161"  # <-- replace with your Kaggle username
KAGGLE_KEY = "538aca55942027ba363d60cd3bb64058"        # <-- replace with your Kaggle API key

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_json.chmod(0o600)
print("Saved credentials to", kaggle_json)


In [ ]:
import os
import zipfile

os.environ.setdefault("KAGGLE_CONFIG_DIR", str(kaggle_dir))


def download(ref: str, dest_folder: str):
    dest = RAW_DIR / dest_folder
    if dest.exists() and any(dest.iterdir()):
        print(f"[skip] {ref} already downloaded -> {dest}")
        return

    dest.mkdir(parents=True, exist_ok=True)

    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    print(f"[download] {ref} -> {dest}")
    api.dataset_download_files(ref, path=str(dest), unzip=False)

    for zip_path in dest.glob("*.zip"):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(dest)
        zip_path.unlink()


download(DATASETS["classification"], "classification")
download(DATASETS["nlp"], "nlp")
download(DATASETS["sentiment"], "sentiment")
download(DATASETS["images"], "images")
print("\nAll datasets downloaded to", RAW_DIR)


## 3. Data Cleansing & EDA

Cleans the three tabular/text datasets and shows the exploratory charts directly below each
cell (nothing is saved to disk — these plots *are* the output).


### 3.1 Classification dataset (symptoms → disease)

In [ ]:
df_cls_raw = pd.read_csv(RAW_DIR / "classification" / "dataset.csv")
severity = pd.read_csv(RAW_DIR / "classification" / "Symptom-severity.csv")

symptom_cols = [c for c in df_cls_raw.columns if c.startswith("Symptom_")]
df_cls_raw[symptom_cols] = df_cls_raw[symptom_cols].apply(lambda c: c.str.strip())
df_cls_raw["Disease"] = df_cls_raw["Disease"].str.strip()

symptom_vocab = sorted(severity["Symptom"].str.strip().unique())
print(f"{len(df_cls_raw)} rows, {df_cls_raw['Disease'].nunique()} diseases, {len(symptom_vocab)} known symptoms")

# one-hot encode: 1 if symptom present anywhere in the row's symptom list
row_symptom_sets = df_cls_raw[symptom_cols].apply(lambda r: set(r.dropna()), axis=1)
onehot = pd.DataFrame(
    {sym: row_symptom_sets.apply(lambda s: int(sym in s)) for sym in symptom_vocab}
)
# NOTE: rows are intentionally repeated in the source data (each disease has several valid
# symptom-set presentations) -- kept as-is so every disease has enough rows for a
# meaningful stratified train/test split.
classification_df = pd.concat([df_cls_raw["Disease"], onehot], axis=1).reset_index(drop=True)
print("processed shape:", classification_df.shape)
classification_df.head()


In [ ]:
plt.figure(figsize=(10, 10))
classification_df["Disease"].value_counts().plot(kind="barh")
plt.title("Classification: cases per disease")
plt.xlabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 8))
onehot.sum().sort_values(ascending=False).head(20).plot(kind="barh")
plt.title("Classification: 20 most common symptoms")
plt.xlabel("count")
plt.tight_layout()
plt.show()


### 3.2 NLP dataset (free-text symptoms → disease)

In [ ]:
nlp_df = pd.read_csv(RAW_DIR / "nlp" / "Symptom2Disease.csv")
nlp_df = nlp_df.drop(columns=[c for c in nlp_df.columns if c.startswith("Unnamed")])
nlp_df["text"] = nlp_df["text"].str.strip().str.lower()
nlp_df["label"] = nlp_df["label"].str.strip()
before = len(nlp_df)
nlp_df = nlp_df.drop_duplicates(subset="text").dropna().reset_index(drop=True)
print(f"{before} -> {len(nlp_df)} rows after dedup, {nlp_df['label'].nunique()} classes")

nlp_df["text_len"] = nlp_df["text"].str.split().apply(len)
nlp_df.head()


In [ ]:
plt.figure(figsize=(8, 8))
nlp_df["label"].value_counts().plot(kind="barh")
plt.title("NLP: symptom descriptions per disease label")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
nlp_df["text_len"].hist(bins=20)
plt.title("NLP: symptom description length (words)")
plt.xlabel("word count")
plt.tight_layout()
plt.show()


### 3.3 Sentiment dataset (drug reviews → Positive / Neutral / Negative)

In [ ]:
CLEAN_RE = re.compile(r"<[^>]+>")


def clean_review(text: str) -> str:
    text = html.unescape(str(text))
    text = CLEAN_RE.sub(" ", text)
    text = text.replace('"', "").replace("&#039;", "'")
    return re.sub(r"\s+", " ", text).strip()


def rating_to_sentiment(rating: int) -> str:
    if rating >= 7:
        return "Positive"
    if rating >= 5:
        return "Neutral"
    return "Negative"


train_raw = pd.read_csv(RAW_DIR / "sentiment" / "drugsComTrain_raw.csv")
test_raw = pd.read_csv(RAW_DIR / "sentiment" / "drugsComTest_raw.csv")
sentiment_all = pd.concat([train_raw, test_raw], ignore_index=True)
print(f"{len(sentiment_all)} raw reviews")

sentiment_all["review"] = sentiment_all["review"].apply(clean_review)
sentiment_all = sentiment_all[sentiment_all["review"].str.split().apply(len) >= 3]
sentiment_all["sentiment"] = sentiment_all["rating"].apply(rating_to_sentiment)
sentiment_all = sentiment_all.drop_duplicates(subset="review").dropna(subset=["review"]).reset_index(drop=True)
print(f"{len(sentiment_all)} rows after cleaning/dedup")
print("class balance before undersampling:")
print(sentiment_all["sentiment"].value_counts())

plt.figure(figsize=(5, 4))
sentiment_all["rating"].value_counts().sort_index().plot(kind="bar")
plt.title("Sentiment: raw star-rating distribution")
plt.tight_layout()
plt.show()


In [ ]:
# balance classes by undersampling the majority classes so training is fast and the
# model isn't just learning to predict "Positive" every time
n_per_class = min(6000, sentiment_all["sentiment"].value_counts().min())
parts = [
    sentiment_all[sentiment_all["sentiment"] == cls].sample(n=n_per_class, random_state=RANDOM_STATE)
    for cls in sentiment_all["sentiment"].unique()
]
sentiment_df = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
sentiment_df = sentiment_df[["drugName", "condition", "review", "rating", "sentiment"]]
print(f"balanced shape: {sentiment_df.shape} ({n_per_class}/class)")

plt.figure(figsize=(5, 4))
sentiment_df["sentiment"].value_counts().plot(kind="bar", color=["#2a9d8f", "#e9c46a", "#e76f51"])
plt.title("Sentiment: balanced class distribution (used for training)")
plt.tight_layout()
plt.show()


## 4. Model Training & Testing

Trains and hyperparameter-tunes all four models. Each section prints its tuned
hyperparameters, test-set accuracy/F1, and a full classification report as plain cell
output (no files written).


### 4.1 Classification — RandomForest on symptoms → disease

In [ ]:
# the source data repeats each unique symptom-combo ~16x on average (4920 rows, only 301
# unique combos). splitting on the raw rows lets identical combos leak into both train and
# test, which is memorization, not generalization -- so we dedupe to unique
# (symptom-set, disease) pairs *before* splitting.
cls_dedup = classification_df.drop_duplicates().reset_index(drop=True)
X, y = cls_dedup.drop(columns=["Disease"]), cls_dedup["Disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [None, 15],
        "min_samples_split": [2, 4],
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy",
)
grid.fit(X_train, y_train)
classification_model = grid.best_estimator_
y_pred = classification_model.predict(X_test)
cls_acc = accuracy_score(y_test, y_pred)
cls_f1 = f1_score(y_test, y_pred, average="macro")

print(f"best params: {grid.best_params_}")
print(f"test accuracy={cls_acc:.4f}  macro-F1={cls_f1:.4f}")
assert cls_acc >= ACCEPTED_ACCURACY["classification"], "classification model below accepted accuracy"
print()
print(classification_report(y_test, y_pred))

symptom_columns = list(X.columns)
classification_classes = sorted(y.unique())
joblib.dump(
    {"model": classification_model, "symptom_columns": symptom_columns, "classes": classification_classes},
    MODELS_DIR / "classification_model.joblib",
)


### 4.2 NLP — TF-IDF + Logistic Regression on free-text symptoms → disease

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    nlp_df["text"], nlp_df["label"], test_size=0.2, random_state=RANDOM_STATE, stratify=nlp_df["label"]
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
grid = GridSearchCV(
    pipe,
    param_grid={
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [1, 2],
        "clf__C": [1, 10, 50],
    },
    cv=5,
    n_jobs=-1,
    scoring="f1_macro",
)
grid.fit(X_train, y_train)
nlp_model = grid.best_estimator_
y_pred = nlp_model.predict(X_test)
nlp_acc = accuracy_score(y_test, y_pred)
nlp_f1 = f1_score(y_test, y_pred, average="macro")

print(f"best params: {grid.best_params_}")
print(f"test accuracy={nlp_acc:.4f}  macro-F1={nlp_f1:.4f}")
assert nlp_acc >= ACCEPTED_ACCURACY["nlp"], "nlp model below accepted accuracy"
print()
print(classification_report(y_test, y_pred))

joblib.dump(nlp_model, MODELS_DIR / "nlp_model.joblib")


### 4.3 Sentiment — TF-IDF + Logistic Regression on drug reviews

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    sentiment_df["review"], sentiment_df["sentiment"], test_size=0.2,
    random_state=RANDOM_STATE, stratify=sentiment_df["sentiment"]
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=20000)),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
grid = GridSearchCV(
    pipe,
    param_grid={
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "clf__C": [1, 5, 10],
    },
    cv=3,
    n_jobs=-1,
    scoring="f1_macro",
)
grid.fit(X_train, y_train)
sentiment_model = grid.best_estimator_
y_pred = sentiment_model.predict(X_test)
sent_acc = accuracy_score(y_test, y_pred)
sent_f1 = f1_score(y_test, y_pred, average="macro")

print(f"best params: {grid.best_params_}")
print(f"test accuracy={sent_acc:.4f}  macro-F1={sent_f1:.4f}")
assert sent_acc >= ACCEPTED_ACCURACY["sentiment"], "sentiment model below accepted accuracy"
print()
print(classification_report(y_test, y_pred))

joblib.dump(sentiment_model, MODELS_DIR / "sentiment_model.joblib")


### 4.4 Deep Learning — transfer-learned ResNet18 on skin-condition images

If you want this to run quickly, enable a GPU first: `Runtime ▸ Change runtime type ▸ T4 GPU`.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

image_data_dir = RAW_DIR / "images" / "Split_smol"
train_tf = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(image_data_dir / "train", transform=train_tf)
val_ds = datasets.ImageFolder(image_data_dir / "val", transform=eval_tf)
image_class_names = train_ds.classes
print(f"{len(train_ds)} train / {len(val_ds)} val images, {len(image_class_names)} classes")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


def build_model(unfreeze_layer4: bool):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in m.parameters():
        param.requires_grad = False
    if unfreeze_layer4:
        for param in m.layer4.parameters():
            param.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, len(image_class_names))
    return m.to(device)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


def run_config(lr, unfreeze_layer4, epochs):
    model = build_model(unfreeze_layer4)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
    val_acc = evaluate(model, val_loader)
    return model, val_acc


# small hyperparameter search: learning rate x how much of the backbone to fine-tune
search_space = [
    {"lr": 1e-3, "unfreeze_layer4": False, "epochs": 8},
    {"lr": 3e-4, "unfreeze_layer4": True, "epochs": 8},
    {"lr": 1e-4, "unfreeze_layer4": True, "epochs": 12},
]

image_model, best_acc, best_cfg = None, -1, None
for cfg in search_space:
    t0 = time.time()
    model, val_acc = run_config(cfg["lr"], cfg["unfreeze_layer4"], cfg["epochs"])
    print(f"config {cfg} -> val_acc={val_acc:.4f} ({time.time()-t0:.1f}s)")
    if val_acc > best_acc:
        image_model, best_acc, best_cfg = model, val_acc, cfg

print(f"\nbest config: {best_cfg}  val_acc={best_acc:.4f}")
assert best_acc >= ACCEPTED_ACCURACY["images"], "image model below accepted accuracy"

image_model.eval()
torch.save(image_model.state_dict(), MODELS_DIR / "image_model.pt")
with open(MODELS_DIR / "image_classes.json", "w") as f:
    json.dump({"classes": image_class_names, "unfreeze_layer4": best_cfg["unfreeze_layer4"]}, f)


### 4.5 Final results summary

In [ ]:
results = {
    "classification": cls_acc,
    "nlp": nlp_acc,
    "sentiment": sent_acc,
    "images": best_acc,
}
print("=== Final test accuracies ===")
for k, v in results.items():
    print(f"  {k:15s}: {v:.4f}  (min accepted {ACCEPTED_ACCURACY[k]})")

pd.DataFrame(
    {"test_accuracy": results, "min_accepted": ACCEPTED_ACCURACY}
).round(4)


## 5. Explainable AI (SHAP)

Explains the classification (RandomForest) model: which symptoms matter most **overall**
(global), and why **one specific patient** got their prediction (local). Plots are shown
inline only.


In [ ]:
explainer = shap.TreeExplainer(classification_model)
shap_sample = X.sample(n=min(150, len(X)), random_state=RANDOM_STATE)
sv = explainer(shap_sample)
print(f"SHAP values shape: {sv.values.shape}")

# ---- global summary: mean |SHAP| per symptom, averaged across all disease classes
mean_abs_per_class = np.abs(sv.values).mean(axis=0)   # (n_features, n_classes)
global_importance = mean_abs_per_class.mean(axis=1)   # (n_features,)
top_idx = np.argsort(-global_importance)[:15]

plt.figure(figsize=(8, 6))
plt.barh(
    [symptom_columns[i] for i in top_idx][::-1],
    global_importance[top_idx][::-1],
    color="#2a9d8f",
)
plt.xlabel("mean |SHAP value| (avg across all diseases)")
plt.title("Which symptoms most influence the disease predictor overall?")
plt.tight_layout()
plt.show()


In [ ]:
def explain_prediction(symptom_vector: pd.DataFrame, top_n: int = 5):
    """symptom_vector: 1-row DataFrame with the same columns as symptom_columns.

    Returns (predicted_disease, [(symptom, shap_value), ...]) for the top_n symptoms
    that pushed the model most strongly toward its prediction.
    """
    sv_one = explainer(symptom_vector)
    pred = classification_model.predict(symptom_vector)[0]
    class_idx = list(classification_model.classes_).index(pred)
    values = sv_one.values[0, :, class_idx]
    order = np.argsort(-np.abs(values))[:top_n]
    contributions = [(symptom_columns[i], float(values[i])) for i in order]
    return pred, contributions


# ---- local explanation: one example patient
example = X.sample(n=1, random_state=RANDOM_STATE)
pred, contributions = explain_prediction(example)
print(f"Example patient -> predicted disease: {pred}\n")
for sym, val in contributions:
    direction = "pushes toward" if val > 0 else "pushes away from"
    print(f"  {sym:30s} {direction} '{pred}'  (shap={val:+.3f})")

class_idx = list(classification_model.classes_).index(pred)
example_sv = explainer(example)[0, :, class_idx]

plt.figure()
shap.plots.waterfall(example_sv, max_display=10, show=False)
plt.title(f"Why this patient was predicted as: {pred}")
plt.tight_layout()
plt.show()


## 6. Live Inference Demo

This replaces the previous Streamlit web app. There's nothing separate to deploy or
run — just fill in the **Colab form fields** below (the ▸ icon on the right of each cell)
and re-run that cell. Everything reuses the models already trained in memory above.


In [ ]:
disease_desc = pd.read_csv(RAW_DIR / "classification" / "symptom_Description.csv")
disease_prec = pd.read_csv(RAW_DIR / "classification" / "symptom_precaution.csv")
disease_desc["key"] = disease_desc["Disease"].str.strip().str.lower()
disease_prec["key"] = disease_prec["Disease"].str.strip().str.lower()
disease_desc = disease_desc.set_index("key")
disease_prec = disease_prec.set_index("key")


def show_disease_info(disease_name: str):
    key = disease_name.strip().lower()
    if key in disease_desc.index:
        print(f"\nℹ️  {disease_desc.loc[key, 'Description']}")
    if key in disease_prec.index:
        precautions = [
            p for p in disease_prec.loc[key, ["Precaution_1", "Precaution_2", "Precaution_3", "Precaution_4"]]
            if isinstance(p, str) and p.strip()
        ]
        if precautions:
            print("Suggested precautions:")
            for p in precautions:
                print(f"  - {p.capitalize()}")


### 6.1 Describe your symptoms (NLP)

In [ ]:
symptom_description = "I've had a burning feeling when I pee and lower back pain for two days"  #@param {type:"string"}

proba = nlp_model.predict_proba([symptom_description.lower()])[0]
classes = nlp_model.classes_
top3_idx = np.argsort(-proba)[:3]
pred = classes[top3_idx[0]]
confidence = proba[top3_idx[0]]

if confidence < CONFIDENCE_THRESHOLDS["nlp"]:
    print("⚠️  Unable to confidently identify a condition from this description. Please contact a doctor.")
    print("\nRaw model output (low confidence - for reference only):")
    for i in top3_idx:
        print(f"  {classes[i]:25s} {proba[i]*100:5.1f}%")
else:
    print(f"✅ Most likely: {pred}  ({confidence*100:.1f}% confidence)\n")
    for i in top3_idx:
        print(f"  {classes[i]:25s} {proba[i]*100:5.1f}%")
    show_disease_info(pred)


### 6.2 Upload a photo of a visible skin condition (Deep Learning)

In [ ]:
from google.colab import files as colab_files

print("Choose an image (jpg/jpeg/png):")
uploaded_img = colab_files.upload()
img_bytes = next(iter(uploaded_img.values()))


In [ ]:
image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title("Uploaded image")
plt.show()

pixel_std = np.array(image.resize((160, 160))).std()
if pixel_std < MIN_IMAGE_STD:
    print("⚠️  This image appears blank or invalid. Please upload a clear photo of the affected skin area, or contact a doctor.")
else:
    infer_tf = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    x = infer_tf(image).unsqueeze(0).to(device)
    image_model.eval()
    with torch.no_grad():
        logits = image_model(x)
        proba = torch.softmax(logits, dim=1)[0].cpu().numpy()

    top3_idx = np.argsort(-proba)[:3]
    confidence = proba[top3_idx[0]]

    print("\n⚠️  Preliminary recognition only -- always consult a dermatologist for an actual diagnosis.")
    print("Known limitation: not hardened against out-of-distribution images -- an unrelated photo")
    print("can still receive a confident-looking (and wrong) prediction (see report for discussion).\n")

    if confidence < CONFIDENCE_THRESHOLDS["images"]:
        print("⚠️  Unable to confidently recognise a condition from this image. Please contact a doctor.")
        print("\nRaw model output (low confidence - for reference only):")
        for i in top3_idx:
            print(f"  {image_class_names[i]:25s} {proba[i]*100:5.1f}%")
    else:
        print(f"✅ Most likely: {image_class_names[top3_idx[0]]}  ({confidence*100:.1f}% confidence)\n")
        for i in top3_idx:
            print(f"  {image_class_names[i]:25s} {proba[i]*100:5.1f}%")


### 6.3 Patient feedback sentiment

In [ ]:
feedback_text = "This medication worked great for me with barely any side effects, would recommend!"  #@param {type:"string"}

proba = sentiment_model.predict_proba([feedback_text])[0]
classes = sentiment_model.classes_
pred = classes[np.argmax(proba)]
confidence = proba.max()

if confidence < CONFIDENCE_THRESHOLDS["sentiment"]:
    print("⚠️  Unable to confidently classify the sentiment of this feedback.")
    print("\nRaw model output (low confidence - for reference only):")
    for c, p in zip(classes, proba):
        print(f"  {c:10s} {p*100:5.1f}%")
else:
    colour = {"Positive": "🟢", "Neutral": "🟡", "Negative": "🔴"}.get(pred, "")
    print(f"✅ Sentiment: {colour} {pred}  ({confidence*100:.1f}% confidence)\n")
    for c, p in zip(classes, proba):
        print(f"  {c:10s} {p*100:5.1f}%")


### 6.4 Symptom checklist (Classification + SHAP explanation)

Enter a comma-separated list of symptoms using the vocabulary the model was trained on
(underscores instead of spaces, e.g. `skin_rash, joint_pain, high_fever`). Run
`print(symptom_vocab)` in a new cell to see the full list.


In [ ]:
symptoms_input = "skin_rash, joint_pain, high_fever, fatigue"  #@param {type:"string"}

requested = [s.strip().lower().replace(" ", "_") for s in symptoms_input.split(",") if s.strip()]
known = [s for s in requested if s in symptom_columns]
unknown = [s for s in requested if s not in symptom_columns]
if unknown:
    print(f"⚠️  Ignoring symptoms not in the training vocabulary: {unknown}")

vector = pd.DataFrame([[int(col in known) for col in symptom_columns]], columns=symptom_columns)
proba = classification_model.predict_proba(vector)[0]
classes = classification_model.classes_
top3_idx = np.argsort(-proba)[:3]
pred = classes[top3_idx[0]]
confidence = proba[top3_idx[0]]

if confidence < CONFIDENCE_THRESHOLDS["classification"]:
    print("⚠️  Unable to confidently predict a condition from these symptoms. Please contact a doctor.")
else:
    print(f"✅ Most likely: {pred}  ({confidence*100:.1f}% confidence)\n")
    for i in top3_idx:
        print(f"  {classes[i]:25s} {proba[i]*100:5.1f}%")
    show_disease_info(pred)

    _, contributions = explain_prediction(vector)
    print("\nWhy this prediction (SHAP):")
    for sym, val in contributions:
        direction = "pushes toward" if val > 0 else "pushes away from"
        print(f"  {sym:25s} {direction} '{pred}'  (shap={val:+.3f})")
